# Preparar y estructurar datos con Python: ejemplos de las lecciones

Ejemplos de la explicación de las lecciones 2, 3 y 4. Los laboratorios guiados y los retos están en `pandas_labs.ipynb`.

Ejecuta las celdas en orden: cada lección usa las variables de la anterior.

## Preparación: carga de los datos crudos

In [20]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')

orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

print(pd.__version__)
print(orders.shape, products.shape, order_prods.shape, aisles.shape, departments.shape)
# Esperado: (478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)

3.0.6
(478967, 6) (49694, 4) (4545007, 4) (134, 2) (21, 2)


# Lección 2: Limpieza y estandarización de datos crudos

Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

### Duplicados exactos: detectar con `duplicated`

In [21]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)
# Esperado: 30 filas, todas con order_dow = 3 y order_hour_of_day = 2

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


### Duplicados exactos: eliminar con `drop_duplicates`

In [22]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)
# Esperado: (478952, 6) True

(478952, 6) True


### Tipos de dato con `astype`

In [23]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')
# Esperado: Antes: 23.0 MB / Después: 7.2 MB

Antes: 23.0 MB
Después: 7.2 MB


### Nulos: comprobar la causa antes de decidir

In [24]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())
# Esperado: True -> cada nulo es un primer pedido: se conserva

True


In [25]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())
# Esperado: una combinación (100, 21) con 1258 productos y luego 0

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

### Texto con los métodos `.str`

In [26]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())
# Esperado: Espacios dobles: 403 / Espacio invisible: 16 / 0 0

Espacios dobles: 403
Espacio invisible: 16
0 0


### Duplicados implícitos

In [27]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))
# Esperado: 141 / 268 de 280 -> conservar y documentar; agrupar por product_id

141
268 de 280


### Limpieza reutilizable con funciones

In [28]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)
# Esperado: (478952, 6)

(478952, 6)


### Verificación de toda la limpieza

In [29]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
# Si la celda no muestra nada, la limpieza está completa

<!-- leccion-3 -->
## Lección 3: Filtrado, ordenamiento y agrupación de datos

Cada pregunta se traduce primero a la frase "*métrica* por *grupo*". Por ejemplo, "¿a qué hora se compra más?" se convierte en "número de pedidos por hora".

Estos ejemplos usan las tablas limpias de la Lección 2 que están más arriba en este cuaderno.

In [30]:
# Comprobación: orders debe venir limpia de la Lección 2
print(orders.shape)
# Esperado: (478952, 6)

(478952, 6)


### Filtrado con máscaras booleanas

In [31]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))
# Esperado: tres valores booleanos y luego 28817

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


Combinación de condiciones: `&` ("y"), `|` ("o") y `~` ("no"). Con máscaras no se usan `and` ni `or`.

In [32]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))
# Esperado: 45599

45599


### Filtrado con `query`

Dentro de `query` sí se usan `and`, `or` y `not`, y los nombres de columna van sin comillas.

In [33]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))
# Esperado: 1449

1449


### Ordenamiento con `sort_values` y `nlargest`

In [34]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))
# Esperado: tres clientes con order_number 100.
# En historial, el cliente 2 aparece primero, con sus pedidos 1 y 4

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


### Agrupación con `groupby`

In [35]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())
# Esperado: 10 40578 (la hora pico es las 10 a. m.)

10 40578


Varias métricas a la vez con **agregación con nombre**:

In [36]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia
# Esperado:
# order_dow  pedidos  clientes  espera_media
# 0          84090    57661     11.8
# 1          82185    56479     11.3
# 2          65833    48587     11.2
# 3          60897    45331     10.7
# 4          59810    44281     10.5
# 5          63488    46127     10.5
# 6          62649    45604     11.4

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


### Tabla dinámica con `pivot_table`

In [37]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]
# Esperado: forma (24, 7). La celda más alta es lunes (1) a las 10, con 7881 pedidos

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


### Valores por fila con `transform`

`transform` calcula el valor del grupo y lo devuelve **en cada fila original**. Esta celda borra al final la columna auxiliar: si la ejecutas dos veces seguidas da `KeyError`. En ese caso, ejecuta el cuaderno desde el principio.

In [38]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')
# Esperado: 231946 30527

231946 30527


### Resumen de la Lección 3

1. Filtrado con máscaras booleanas y los operadores `&`, `|` y `~`.
2. Filtros `isin`, `between` y `query`.
3. Ordenamiento con `sort_values`, `nlargest` y `nsmallest`.
4. Agrupación con `groupby` y agregación con nombre `agg`.
5. Tablas dinámicas con `pivot_table`.
6. Valores de grupo por fila con `transform`.